# 1. INTRODUCCIÓN

Este notebook forma parte de la fase de **procesamiento distribuido y generación de variables derivadas** dentro del pipeline global de datos del proyecto. Su objetivo es transformar el dataset histórico previamente construido en un conjunto de datos enriquecido, optimizado y listo para su uso en tareas de **machine learning y análisis avanzado de series temporales**.

A diferencia de la fase anterior, centrada en la extracción, limpieza y almacenamiento en un Data Lake, en este módulo se introduce el **uso de computación distribuida mediante Apache Spark**, lo que permite escalar el procesamiento de datos y operar eficientemente sobre volúmenes elevados de información.

El entorno se configura a través de una instancia local de Spark, ejecutando un clúster en modo `local[*]`, lo que habilita el uso paralelo de los núcleos del procesador para simular un entorno distribuido. Sobre este entorno se carga el dataset consolidado de criptomonedas previamente almacenado en formato Parquet.

El conjunto de datos está compuesto por información histórica de los principales activos del mercado: Bitcoin, Ethereum, Solana, BNB y XRP, integrando datos de mercado y capitalización procedentes de fuentes como Binance y CoinGecko.

Una vez cargados los datos, se inicia una fase de feature engineering, en la que se eliminan variables redundantes y se construyen nuevas métricas clave para el análisis predictivo. Entre ellas destacan:

* Variaciones porcentuales a corto, medio y largo plazo (1h, 24h y 7d).
* Indicadores de volatilidad basados en ventanas móviles.
* Cálculo del RSI (Relative Strength Index) como indicador técnico de momentum.
* Agregación de volumen en ventanas de 24 horas.

Estas transformaciones se realizan mediante **ventanas temporales (window functions)** de Spark, lo que permite calcular métricas dependientes del orden temporal de forma eficiente y escalable.

El resultado es un dataset estructurado a nivel de observación temporal por criptomoneda, enriquecido con variables derivadas que capturan patrones de comportamiento del mercado y que resultan fundamentales para tareas de predicción y modelado.

Finalmente, el dataset procesado se almacena en formato **Parquet optimizado para machine learning**, generando una versión final lista para su integración en modelos predictivos.

Al igual que en fases anteriores, este notebook se ha desarrollado como **entorno de validación del pipeline de procesamiento distribuido**, con el objetivo de comprobar la correcta implementación de las transformaciones y su posterior traslado a scripts modulares dentro de una arquitectura más robusta y automatizada orientada a producción.

# 2. CONFIGURACIÓN DEL ENTORNO DISTRIBUIDO

In [1]:
import os
import findspark
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-21"
os.environ["PYSPARK_PYTHON"] = "python"

findspark.init()

# Crear una SparkSession para analizar datos a gran escala, iniciando un clúster local que utiliza los núcleos de la CPU para el procesamiento distribuido
spark = SparkSession.builder.master("local[*]").appName("MotorProcesamientoCriptomonedas").getOrCreate()
print("\nClúster de Spark iniciado con éxito. ✅")
print(f"Spark version: {spark.version}")

# Cargar el fichero 'all_cryptos.parquet'
df_cryptos = spark.read.parquet("../data/processed/data_lake/all_cryptos.parquet")
df_cryptos.printSchema()
print("\nDatos cargados correctamente. ✅")


Clúster de Spark iniciado con éxito. ✅
Spark version: 4.1.1
root
 |-- crypto: string (nullable = true)
 |-- open_time: timestamp_ntz (nullable = true)
 |-- open_price: double (nullable = true)
 |-- high_price: double (nullable = true)
 |-- low_price: double (nullable = true)
 |-- close_price: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- close_time: timestamp_ntz (nullable = true)
 |-- market_cap: double (nullable = true)


Datos cargados correctamente. ✅


# 3. CÁLCULO DE MÉTRICAS (FEATURE ENGINEERING)

In [2]:
from pyspark.sql.functions import col, lag, when, avg, stddev, sum
from pyspark.sql.window import Window

# Eliminación de features irrelevantes
df_cryptos = df_cryptos.drop("open_price", "high_price", "low_price", "close_time")

# Window por crypto, ordenado por tiempo
w = Window.partitionBy("crypto").orderBy("open_time")

# change_1h_pct — variación respecto a la vela anterior (1h)
df_cryptos = df_cryptos.withColumn("change_1h_pct",
    (col("close_price") - lag("close_price", 1).over(w)) / lag("close_price", 1).over(w) * 100
)

# change_24h_pct — 24 velas atrás (24h)
df_cryptos = df_cryptos.withColumn("change_24h_pct",
    (col("close_price") - lag("close_price", 24).over(w)) / lag("close_price", 24).over(w) * 100
)

# change_7d_pct — 168 velas atrás (7 días * 24h)
df_cryptos = df_cryptos.withColumn("change_7d_pct",
    (col("close_price") - lag("close_price", 168).over(w)) / lag("close_price", 168).over(w) * 100 
)

# volatility_30d — std de los últimos 30 días (720 velas)
w_30d = Window.partitionBy("crypto").orderBy("open_time").rowsBetween(-719, 0)
df_cryptos = df_cryptos.withColumn("volatility_30d", stddev("close_price").over(w_30d))

# RSI_14 — requiere cálculo de gains/losses
df_cryptos = df_cryptos.withColumn("price_diff", col("close_price") - lag("close_price", 1).over(w))
df_cryptos = df_cryptos.withColumn("gain", when(col("price_diff") > 0, col("price_diff")).otherwise(0))
df_cryptos = df_cryptos.withColumn("loss", when(col("price_diff") < 0, -col("price_diff")).otherwise(0))

w_14 = Window.partitionBy("crypto").orderBy("open_time").rowsBetween(-13, 0)
df_cryptos = df_cryptos.withColumn("avg_gain", avg("gain").over(w_14))
df_cryptos = df_cryptos.withColumn("avg_loss", avg("loss").over(w_14))
df_cryptos = df_cryptos.withColumn("rsi_14",
    when(col("avg_loss") == 0, 100)
    .otherwise(100 - (100 / (1 + col("avg_gain") / col("avg_loss"))))
)

# volume_24h - volumen acumulado de las últimas 24 horas
w_24h = Window.partitionBy("crypto").orderBy("open_time").rowsBetween(-23, 0)
df_cryptos = df_cryptos.withColumn("volume_24h", sum("volume").over(w_24h))

# Limpiamos columnas intermedias
df_cryptos = df_cryptos.drop("volume", "price_diff", "gain", "loss", "avg_gain", "avg_loss")

# Renombramos filas para tener el dataset lo más preciso posible con respecto al modelo de datos
renames = {
    "open_time": "timestamp",
    "close_price": "price_usd",
}

for old_name, new_name in renames.items():
    df_cryptos = df_cryptos.withColumnRenamed(old_name, new_name)

print("\n---- FASE DE CÁLCULO DE MÉTRICAS COMPLETADA ✅ ----")
print("\nDataFrame resultante:")
df_cryptos.printSchema()
df_cryptos.show(10, truncate=False)


---- FASE DE CÁLCULO DE MÉTRICAS COMPLETADA ✅ ----

DataFrame resultante:
root
 |-- crypto: string (nullable = true)
 |-- timestamp: timestamp_ntz (nullable = true)
 |-- price_usd: double (nullable = true)
 |-- market_cap: double (nullable = true)
 |-- change_1h_pct: double (nullable = true)
 |-- change_24h_pct: double (nullable = true)
 |-- change_7d_pct: double (nullable = true)
 |-- volatility_30d: double (nullable = true)
 |-- rsi_14: double (nullable = true)
 |-- volume_24h: double (nullable = true)

+-----------+-------------------+---------+--------------------+---------------------+--------------+-------------+------------------+------------------+------------------+
|crypto     |timestamp          |price_usd|market_cap          |change_1h_pct        |change_24h_pct|change_7d_pct|volatility_30d    |rsi_14            |volume_24h        |
+-----------+-------------------+---------+--------------------+---------------------+--------------+-------------+------------------+--------

# 4. ALMACENAMIENTO DE DATASET FINAL

In [3]:
output_dir = r"C:\Users\Usuario\Desktop\tfm-ia\data\ml\features"
os.makedirs(output_dir, exist_ok=True)

# Almacenar en formato .parquet el dataset resultante final listo para ML
df_cryptos.toPandas().to_parquet(f"{output_dir}/all_cryptos_ml.parquet", index=False)

print("\n---- 🧪 DATASET FINAL PREPARADO PARA MACHINE LEARNING ALMACENADO CORRECTAMENTE 🧪 ----")


---- 🧪 DATASET FINAL PREPARADO PARA MACHINE LEARNING ALMACENADO CORRECTAMENTE 🧪 ----
